# Bayesian linear & logistic regression — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def normalize(v):
    v=np.asarray(v,dtype=float); return v/v.sum()
def softmax(v):
    v=np.asarray(v,dtype=float); e=np.exp(v-v.max()); return e/e.sum()
def norm_pdf(x,mu,var):
    return np.exp(-0.5*(np.asarray(x)-mu)**2/var)/np.sqrt(2*np.pi*var)
def beta_pdf_grid(x,a,b):
    B=math.gamma(a)*math.gamma(b)/math.gamma(a+b)
    return (np.asarray(x)**(a-1))*((1-np.asarray(x))**(b-1))/B
print('setup ok')

## Regression weights become uncertain quantities with a posterior mean and variance
Bayesian regression treats coefficients as random variables. These five examples compute a one-weight Gaussian posterior, show how noise and priors move it, predict with uncertainty, and contrast a logistic grid posterior.

In [ ]:
# 1) one-weight Bayesian linear regression posterior
x=np.array([1.,2.,3.]); y=np.array([1.2,1.9,3.2]); tau2=4.; sig2=0.25
prec=1/tau2 + (x@x)/sig2; var=1/prec; mean=var*(x@y)/sig2
w=np.linspace(0,2,400); plt.figure(figsize=(6,3)); plt.plot(w,norm_pdf(w,mean,var)); plt.axvline(mean,c='r',ls='--'); plt.title(f'posterior w ~ N({mean:.3f}, {var:.3f})')
assert abs(mean-1.0382222222222224)<1e-12 and abs(var-0.017777777777777778)<1e-12

In [ ]:
# 2) more observation noise widens the weight posterior
sigs=[0.1,0.25,1.0]; vals=[]
for s2 in sigs:
    v=1/(1/tau2+(x@x)/s2); vals.append(v)
plt.figure(figsize=(6,3)); plt.plot(sigs,vals,'-o'); plt.xlabel('noise variance'); plt.ylabel('posterior variance'); plt.title('noisier data leaves wider belief')
assert vals[2]>vals[0]

In [ ]:
# 3) stronger prior shrinkage pulls the mean toward zero
priors=[0.25,1,4,100]; means=[]
for t2 in priors:
    v=1/(1/t2+(x@x)/sig2); means.append(v*(x@y)/sig2)
plt.figure(figsize=(6,3)); plt.plot(priors,means,'-o'); plt.xscale('log'); plt.xlabel('prior variance'); plt.ylabel('posterior mean'); plt.title('tight priors shrink coefficients')
assert means[0]<means[-1]

In [ ]:
# 4) predictive distribution combines weight uncertainty and noise
xstar=np.array([0.,1.,2.,4.]); pred_mean=xstar*mean; pred_var=sig2 + xstar**2*var
plt.figure(figsize=(6,3)); plt.errorbar(xstar,pred_mean,yerr=2*np.sqrt(pred_var),fmt='o-'); plt.xlabel('x*'); plt.ylabel('y*'); plt.title('prediction intervals widen away from zero')
assert abs(pred_mean[-1]-4.15288888888889)<1e-12 and pred_var[-1]>pred_var[1]

In [ ]:
# 5) logistic regression posterior by grid: likelihood times Gaussian prior
xx=np.array([-2.,-1.,1.,2.]); yy=np.array([0,0,1,1]); grid=np.linspace(-1,4,500)
logp=-0.5*grid**2/4
for xi,yi in zip(xx,yy):
    p=1/(1+np.exp(-grid*xi)); logp += yi*np.log(p)+(1-yi)*np.log(1-p)
p=softmax(logp); m=(grid*p).sum()
plt.figure(figsize=(6,3)); plt.plot(grid,p); plt.axvline(m,c='r',ls='--'); plt.title(f'logistic posterior mean ≈ {m:.3f}')
assert 1.0<m<2.0